# Imports

In [1]:
from pathlib import Path
print(Path.cwd())

import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent)) # problem with dependency resolution (e.g. custom_builder)without this

/Users/mac/Documents/dev/ID2221/dic/Week 2


In [2]:
# Use delta features if needed (DeltaTable, etc.)
from delta import *
from custom_builder import builder
from log import *

# use the existing preconfigured builder to create the Spark session.
spark = configure_spark_with_delta_pip(builder).getOrCreate()

print(f'current database: {spark.catalog.currentDatabase()}')
print(f'spark tables: {spark.catalog.listTables()}')

from pyspark.sql import functions as F

import json

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/15 17:00:34 WARN Utils: Your hostname, MacBook-Pro-som-tillhor-MAC.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.247 instead (on interface en0)
26/09/15 17:00:34 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/Users/mac/Documents/dev/ID2221/dic/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/mac/.ivy2.5.2/cache
The jars for the packages stored in: /Users/mac/.ivy2.5.2/jars
io.delta#delta-spark_4.2_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-04ff5dee-0f9c-4b7c-810b-89f87b468232;1.0
	confs: [default]
	found io.delta#delta-spark_4.2_2.13;4.4.0 in central
	found io.delta#delta-storage;4.4.0 in central
	found io.unitycatalog#unitycatalog-client;0.6.0 in central
	found org.slf4j#slf4j-ap

current database: default
spark tables: [Table(name='air_quality', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False), Table(name='integrated_taxi_trips', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False), Table(name='taxi_trips', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False), Table(name='taxi_zone_lookup', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False), Table(name='weather', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False)]


# Integrated Table Queries

## Task 1.1

In [3]:
result = spark.sql("""
    SELECT
        pu_zone,
        month(pu_datetime) AS month, 
        COUNT(*) AS row_count
    FROM default.integrated_taxi_trips
    GROUP BY pu_zone, month
""")
result.summary().show()
result.show()

result[result["pu_zone"] == "Van Cortlandt Park"].show()

26/09/15 17:00:45 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+--------------------+-------------------+------------------+
|summary|             pu_zone|              month|         row_count|
+-------+--------------------+-------------------+------------------+
|  count|                 259|                260|               260|
|   mean|                NULL| 1.0115384615384615|11402.126923076923|
| stddev|                NULL|0.10700168900237719| 27833.34443580034|
|    min|Allerton/Pelham G...|                  1|                 1|
|    25%|                NULL|                  1|                91|
|    50%|                NULL|                  1|               258|
|    75%|                NULL|                  1|              1811|
|    max|      Yorkville West|                  2|            145240|
+-------+--------------------+-------------------+------------------+

+--------------------+-----+---------+
|             pu_zone|month|row_count|
+--------------------+-----+---------+
|  Van Cortlandt Park|    1|       18|
|  

In [4]:
result = spark.sql("""
    SELECT
        pu_zone,
        month(pu_datetime) AS month
    FROM default.integrated_taxi_trips
""")

# get dataframe number of rows for Van Cortlandt Park
result[result["pu_zone"] == "Van Cortlandt Park"].count()

18

## Task 1.2

In [5]:
result = spark.sql(
"""
SELECT
    CASE
        WHEN prcp > 0 THEN 'greater_than_0'
        ELSE 'zero_or_null'
  	END AS column_group,
    COUNT(*) AS cnt,
    AVG(trip_distance) AS avg_trip_distance
FROM integrated_taxi_trips
GROUP BY
    CASE
        WHEN prcp > 0 THEN 'greater_than_0'
            ELSE 'zero_or_null'
    END;
"""
)
result.show()


+--------------+-------+-----------------+
|  column_group|    cnt|avg_trip_distance|
+--------------+-------+-----------------+
|greater_than_0| 424773|3.470743355446981|
|  zero_or_null|2539780|3.682436765606291|
+--------------+-------+-----------------+



In [6]:
# select max and min of prcp
result = spark.sql("""
SELECT
    MAX(prcp) AS max_prcp,
    MIN(prcp) AS min_prcp
FROM integrated_taxi_trips
""")
result.show()

+--------+--------+
|max_prcp|min_prcp|
+--------+--------+
|     9.9|     0.0|
+--------+--------+



In [7]:
result = spark.sql(
"""
SELECT
    FLOOR(prcp) AS prcp_interval,
    COUNT(*) AS cnt,
    AVG(trip_distance) AS avg_trip_distance
FROM integrated_taxi_trips
GROUP BY FLOOR(prcp)
ORDER BY prcp_interval;
"""
)
result.show()

+-------------+-------+------------------+
|prcp_interval|    cnt| avg_trip_distance|
+-------------+-------+------------------+
|         NULL| 322358| 3.812452316575689|
|            0|2519587|  3.64919207074348|
|            1|  68492|3.2953087971315957|
|            2|  23104|3.0427263657823804|
|            3|  21631| 3.022273587578323|
|            4|    410|  5.96612194082359|
|            5|   1830| 4.711049171943278|
|            6|   3988|3.7155416268321013|
|            8|    324| 7.272222214010487|
|            9|   2829| 4.881074589306141|
+-------------+-------+------------------+



## Task 1.3

In [8]:
res = spark.sql("""
SELECT measurement, COUNT(pu_county) AS trips
    FROM integrated_taxi_trips
    WHERE measurement IS NOT NULL
    GROUP BY measurement
    ORDER BY measurement, trips
;
""")
res.show()

+-----------+-----+
|measurement|trips|
+-----------+-----+
|        1.3|   20|
|        1.6|   13|
|        1.7|   31|
|        1.8|   41|
|        1.9|   19|
|        2.0|   24|
|        2.1|  135|
|        2.2|   39|
|        2.3|   36|
|        2.4|   51|
|        2.5|  137|
|        2.6|   93|
|        2.7|   84|
|        2.8|   27|
|        2.9|  141|
|        3.0|   62|
|        3.1|   89|
|        3.2|  115|
|        3.3|   71|
|        3.4|  167|
+-----------+-----+
only showing top 20 rows


## Task 1.4

In [9]:
res = spark.sql("""
WITH 
    weather_cat AS (
        SELECT pu_county, DATE_TRUNC('hour', pu_datetime) AS hour,
            CASE
                WHEN prcp > 3 THEN 'rain'
                WHEN temp > 25 THEN 'heatwave'
                WHEN temp < 0 THEN 'cold'
                WHEN rhum > 65 THEN 'humid'
                WHEN rhum < 25 THEN 'dry'
                WHEN wspd > 8 THEN 'stormy'
                ELSE 'moderate'
            END AS weather_cond
        FROM integrated_taxi_trips
        WHERE pu_county IS NOT NULL
    ),
    demand AS (
        SELECT weather_cond, pu_county, COUNT(*) AS trips, COUNT(DISTINCT hour) AS weather_hours
        FROM weather_cat
        GROUP BY weather_cond, pu_county
    )
SELECT weather_cond, pu_county, trips, weather_hours, ROUND(CAST(trips AS DOUBLE) / NULLIF(weather_hours, 0), 2) AS trips_per_hour
    FROM demand
    ORDER BY pu_county, trips_per_hour
;
""")
# note: this is a flawed comparison, as what counts as 'weather condition' is subjective
#       e.g what temperature counts as a 'heatwave'?
res.show()

+------------+---------+-------+-------------+--------------+
|weather_cond|pu_county|  trips|weather_hours|trips_per_hour|
+------------+---------+-------+-------------+--------------+
|         dry|    Bronx|     12|            2|           6.0|
|        rain|    Bronx|     89|           10|           8.9|
|       humid|    Bronx|   2094|          210|          9.97|
|        cold|    Bronx|   1743|          173|         10.08|
|      stormy|    Bronx|   2707|          259|         10.45|
|    moderate|    Bronx|    260|           24|         10.83|
|        rain| Brooklyn|    258|           11|         23.45|
|         dry| Brooklyn|     47|            2|          23.5|
|    moderate| Brooklyn|    849|           26|         32.65|
|       humid| Brooklyn|   7516|          224|         33.55|
|      stormy| Brooklyn|  10103|          294|         34.36|
|        cold| Brooklyn|   6482|          186|         34.85|
|        rain|Manhattan|  18909|           11|        1719.0|
|    mod

## Task 1.5

In [10]:
res = spark.sql("""
SELECT date_format(pu_datetime, 'EEE') AS day, hour(pu_datetime) AS hour, COUNT(*) AS trips
    FROM taxi_trips
    GROUP BY hour, day
    ORDER BY day, trips DESC
;
""")
res.show()

+---+----+-----+
|day|hour|trips|
+---+----+-----+
|Fri|  18|29050|
|Fri|  17|28034|
|Fri|  19|26656|
|Fri|  16|25645|
|Fri|  15|25578|
|Fri|  14|24746|
|Fri|  22|24015|
|Fri|  13|22190|
|Fri|  23|22100|
|Fri|  21|21976|
|Fri|  20|21168|
|Fri|  12|21024|
|Fri|  11|19852|
|Fri|  10|19559|
|Fri|   9|18192|
|Fri|   8|17323|
|Fri|   7|13043|
|Fri|   0| 8804|
|Fri|   6| 6283|
|Fri|   1| 4805|
+---+----+-----+
only showing top 20 rows


## Task 1.6

In [11]:
res = spark.sql("""
SELECT date_format(pu_datetime, 'MMM') AS month, COUNT(*) AS trips
    FROM taxi_trips
    GROUP BY month
    ORDER BY month
;
""")
res.show()

+-----+-------+
|month|  trips|
+-----+-------+
|  Dec|     12|
|  Feb|      3|
|  Jan|2964553|
+-----+-------+



# Underlying Table queries

## Task 2.1

In [4]:
result = spark.sql("""
    SELECT
        zone AS pu_zone,
        month(pu_datetime) AS month, 
        COUNT(*) AS row_count
    FROM default.taxi_trips
    LEFT JOIN default.taxi_zone_lookup
    ON taxi_trips.pu_location_id = taxi_zone_lookup.location_id
    GROUP BY pu_zone, month
""")
result.summary().show()
result.show()

+-------+--------------------+------------------+------------------+
|summary|             pu_zone|             month|         row_count|
+-------+--------------------+------------------+------------------+
|  count|                 270|               271|               271|
|   mean|                NULL|1.4575645756457565|10939.365313653136|
| stddev|                NULL|2.1749942465823655|27353.533786659147|
|    min|Allerton/Pelham G...|                 1|                 1|
|    25%|                NULL|                 1|                78|
|    50%|                NULL|                 1|               245|
|    75%|                NULL|                 1|              1393|
|    max|      Yorkville West|                12|            145240|
+-------+--------------------+------------------+------------------+

+--------------------+-----+---------+
|             pu_zone|month|row_count|
+--------------------+-----+---------+
|  Van Cortlandt Park|    1|       18|
|              

In [5]:
result = spark.sql("""
    SELECT
        zone AS pu_zone,
        month(pu_datetime) AS month
    FROM default.taxi_trips
    LEFT JOIN default.taxi_zone_lookup
    ON taxi_trips.pu_location_id = taxi_zone_lookup.location_id
""")
# get dataframe number of rows for Van Cortlandt Park
result[result["pu_zone"] == "Van Cortlandt Park"].count()

18

In [6]:
# count number of rows in integrated_taxi_trips
result = spark.sql("""
    SELECT COUNT(*) AS row_count
    FROM default.integrated_taxi_trips
""")
result.show()

# count number of rows in taxi_trips
result = spark.sql("""
    SELECT COUNT(*) AS row_count
    FROM default.taxi_trips
""")
result.show()

+---------+
|row_count|
+---------+
|  2964553|
+---------+

+---------+
|row_count|
+---------+
|  2964568|
+---------+

